# EDA interactivo curado para el proyecto de churn

Esta versión está más enfocada a la **defensa** y a la futura app de **Streamlit**. He quitado parte del ruido exploratorio inicial y he dejado los gráficos que mejor cuentan la historia del churn: **tipo de suscripción, engagement, fricción, segmentos y geografía**.

## 1. Imports y carga de datos

In [1]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)


In [2]:
path = "../data/raw/train.csv"

In [3]:


df = pd.read_csv(path).copy()
df.head()


,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,average_session_length,song_skip_rate,weekly_songs_played,weekly_unique_songs,num_favorite_artists,num_platform_friends,num_playlists_created,num_shared_playlists,notifications_clicked,churned
0,1,32,Montana,Free,Yearly,2,Paypal,Medium,-1606,22.391362,105.394516,0.176873,169,109,18,32,52,35,46,0
1,2,64,New Jersey,Free,Monthly,3,Paypal,Low,-2897,29.294210,52.501115,0.981811,55,163,44,33,12,25,37,1
2,3,51,Washington,Premium,Yearly,2,Credit Card,High,-348,15.400312,24.703696,0.048411,244,117,20,129,50,28,38,0
3,4,63,California,Family,Yearly,4,Apple Pay,Medium,-2894,22.842084,83.595480,0.035691,442,252,47,120,55,17,24,0
4,5,54,Washington,Family,Monthly,3,Paypal,High,-92,23.151163,52.578266,0.039738,243,230,41,66,40,32,47,0


## 2. Funciones de preparación

Estas funciones dejan el dataset listo para el EDA sin mezclar todavía la parte de modelado.

In [59]:

def clean_streaming_churn_df(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df = df.copy()

    # Variables derivadas más interpretables
    if "signup_date" in df.columns:
        # En este dataset signup_date aparece en negativo, así que lo convertimos a antigüedad positiva.
        df["tenure_days"] = -df["signup_date"]
        df["tenure_months"] = df["tenure_days"] / 30

    if "weekly_hours" in df.columns and "weekly_songs_played" in df.columns:
        df["songs_per_hour"] = np.where(
            df["weekly_hours"] > 0,
            df["weekly_songs_played"] / df["weekly_hours"],
            np.nan
        )

    if "song_skip_rate" in df.columns:
        df["high_skip_user"] = (df["song_skip_rate"] >= 0.7).astype(int)

    return df


STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebrasksa": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY"
}

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "signup_date" in df.columns:
        df["tenure_days"] = -df["signup_date"]

    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[18, 25, 35, 50, 65, 80],
            labels=["18-24", "25-34", "35-49", "50-64", "65-79"],
            include_lowest=True
        )

    if "weekly_hours" in df.columns:
        df["weekly_hours_bin"] = pd.cut(
            df["weekly_hours"],
            bins=[0, 5, 10, 20, 30, 40, 50],
            labels=["0-5", "5-10", "10-20", "20-30", "30-40", "40-50"],
            include_lowest=True
        )

    if "song_skip_rate" in df.columns:
        df["skip_rate_bin"] = pd.cut(
            df["song_skip_rate"],
            bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
            labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"],
            include_lowest=True
        )

    if {"subscription_type", "customer_service_inquiries"}.issubset(df.columns):
        conditions = [
            (df["subscription_type"] == "Free") & (df["customer_service_inquiries"] == "High"),
            (df["subscription_type"] == "Free"),
            (df["customer_service_inquiries"] == "High")
        ]
        choices = ["Very High Risk Segment", "High Risk Segment", "Medium Risk Segment"]
        df["risk_segment_rule"] = np.select(conditions, choices, default="Lower Risk Segment")

    if "location" in df.columns:
        df["state_code"] = df["location"].map(STATE_ABBREV)

    return df

df = clean_streaming_churn_df(df, is_train=True)
df = add_features(df)
df.head()


,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,average_session_length,song_skip_rate,weekly_songs_played,weekly_unique_songs,num_favorite_artists,num_platform_friends,num_playlists_created,num_shared_playlists,notifications_clicked,churned,tenure_days,tenure_months,songs_per_hour,high_skip_user,age_group,weekly_hours_bin,skip_rate_bin,risk_segment_rule,state_code
0,1,32,Montana,Free,Yearly,2,Paypal,Medium,-1606,22.391362,105.394516,0.176873,169,109,18,32,52,35,46,0,1606,53.533333,7.547554,0,25-34,20-30,0-0.2,High Risk Segment,MT
1,2,64,New Jersey,Free,Monthly,3,Paypal,Low,-2897,29.294210,52.501115,0.981811,55,163,44,33,12,25,37,1,2897,96.566667,1.877504,1,50-64,20-30,0.8-1.0,High Risk Segment,NJ
2,3,51,Washington,Premium,Yearly,2,Credit Card,High,-348,15.400312,24.703696,0.048411,244,117,20,129,50,28,38,0,348,11.600000,15.843835,0,50-64,10-20,0-0.2,Medium Risk Segment,WA
3,4,63,California,Family,Yearly,4,Apple Pay,Medium,-2894,22.842084,83.595480,0.035691,442,252,47,120,55,17,24,0,2894,96.466667,19.350248,0,50-64,20-30,0-0.2,Lower Risk Segment,CA
4,5,54,Washington,Family,Monthly,3,Paypal,High,-92,23.151163,52.578266,0.039738,243,230,41,66,40,32,47,0,92,3.066667,10.496233,0,50-64,20-30,0-0.2,Medium Risk Segment,WA


## 3. Vista general rápida

In [52]:

def basic_overview(df: pd.DataFrame, name: str = "DataFrame") -> None:
    print(f"Shape de {name}: {df.shape}")
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isna().sum())
    if "churned" in df.columns:
        print(f"\nTasa global de churn: {df['churned'].mean():.4f}")


basic_overview(df, name="train_clean")


Shape de train_clean: (125000, 28)

Tipos de datos:
customer_id                      int64
age                              int64
location                           str
subscription_type                  str
payment_plan                       str
num_subscription_pauses          int64
payment_method                     str
customer_service_inquiries         str
signup_date                      int64
weekly_hours                   float64
average_session_length         float64
song_skip_rate                 float64
weekly_songs_played              int64
weekly_unique_songs              int64
num_favorite_artists             int64
num_platform_friends             int64
num_playlists_created            int64
num_shared_playlists             int64
notifications_clicked            int64
churned                          int64
tenure_days                      int64
tenure_months                  float64
songs_per_hour                 float64
high_skip_user                   int64
age_group   

## 4. Funciones de visualización interactivas

In [49]:

def churn_rate(df, target="churned"):
    return df[target].mean() * 100


def plot_global_churn(df, target="churned"):
    churn_pct = df[target].mean() * 100
    no_churn_pct = 100 - churn_pct

    plot_df = pd.DataFrame({
        "Estado": ["No churn", "Churn"],
        "Porcentaje": [no_churn_pct, churn_pct]
    })

    fig = px.bar(
        plot_df,
        x="Estado",
        y="Porcentaje",
        text="Porcentaje",
        title="Distribución global del churn"
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        yaxis_title="Porcentaje",
        xaxis_title="",
        title_x=0.5
    )
    fig.show()


def plot_churn_by_category(df, category_col, target="churned", sort_desc=True, title=None):
    plot_df = (
        df.groupby(category_col)[target]
        .mean()
        .reset_index()
        .assign(churn_pct=lambda x: x[target] * 100)
        [[category_col, "churn_pct"]]
    )

    if sort_desc:
        plot_df = plot_df.sort_values("churn_pct", ascending=False)

    fig = px.bar(
        plot_df,
        x=category_col,
        y="churn_pct",
        text="churn_pct",
        title=title if title else f"Tasa de churn por {category_col}"
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        yaxis_title="Churn (%)",
        xaxis_title=category_col,
        title_x=0.5
    )
    fig.show()

    return plot_df


def plot_churn_by_numeric_bins(df, numeric_col, bin_col, target="churned", title=None):
    plot_df = (
        df.groupby(bin_col, observed=False)[target]
        .mean()
        .reset_index()
        .assign(churn_pct=lambda x: x[target] * 100)
        [[bin_col, "churn_pct"]]
    )

    fig = px.bar(
        plot_df,
        x=bin_col,
        y="churn_pct",
        text="churn_pct",
        title=title if title else f"Tasa de churn por intervalos de {numeric_col}"
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        yaxis_title="Churn (%)",
        xaxis_title=numeric_col,
        title_x=0.5
    )
    fig.show()

    return plot_df


def plot_churn_heatmap(df, row_col, col_col, target="churned", title=None):
    heatmap_df = (
        df.groupby([row_col, col_col])[target]
        .mean()
        .reset_index()
        .assign(churn_pct=lambda x: x[target] * 100)
        .pivot(index=row_col, columns=col_col, values="churn_pct")
    )

    fig = px.imshow(
        heatmap_df,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="Reds",
        title=title if title else f"Heatmap de churn (%) entre {row_col} y {col_col}",
        labels=dict(color="Churn (%)")
    )
    fig.update_layout(title_x=0.5)
    fig.show()

    return heatmap_df


def plot_distribution_by_churn(df, numeric_col, target="churned", nbins=30, title=None):
    df_copy = df.copy()
    df_copy["churn_label"] = df_copy[target].map({0: "No churn", 1: "Churn"})

    fig = px.histogram(
        df_copy,
        x=numeric_col,
        color="churn_label",
        barmode="overlay",
        nbins=nbins,
        opacity=0.6,
        title=title if title else f"Distribución de {numeric_col} según churn"
    )

    fig.update_layout(
        xaxis_title=numeric_col,
        yaxis_title="Frecuencia",
        title_x=0.5
    )
    fig.show()


## 5. Gráficos principales del EDA

Estos son los gráficos que más merecen la pena para la defensa y para reutilizar luego en Streamlit.

In [7]:
print(f"Tasa global de churn: {churn_rate(df):.2f}%")
plot_global_churn(df)

Tasa global de churn: 51.34%


### 5.1 Tipo de suscripción

Este gráfico suele ser uno de los más fuertes del proyecto: permite ver si el compromiso económico con la plataforma se asocia con el churn.

In [8]:
sub_table = plot_churn_by_category(df, "subscription_type", title="Tasa de churn por tipo de suscripción")
sub_table

,subscription_type,churn_pct
1,Free,79.407720
3,Student,57.393388
0,Family,34.580973
2,Premium,33.909549


### 5.2 Fricción del usuario

`customer_service_inquiries` captura muy bien una dimensión de fricción o mala experiencia de usuario.

In [9]:
cs_table = plot_churn_by_category(df, "customer_service_inquiries", title="Tasa de churn por incidencias / consultas")
cs_table

,customer_service_inquiries,churn_pct
0,High,74.333261
2,Medium,50.917100
1,Low,28.923172


### 5.3 Engagement semanal

Las horas semanales son una de las variables numéricas más potentes del dataset.

In [10]:
wh_table = plot_churn_by_numeric_bins(df, "weekly_hours", "weekly_hours_bin", title="Tasa de churn por intervalos de weekly_hours")
wh_table

,weekly_hours_bin,churn_pct
0,0-5,89.121877
1,5-10,72.543076
2,10-20,49.227861
3,20-30,50.058442
4,30-40,49.388077
5,40-50,27.253099


### 5.4 Pausas en la suscripción

Las pausas actúan como señal de inestabilidad. Es un gráfico sencillo pero muy explicable en la defensa.

In [11]:
pauses_table = plot_churn_by_category(df, "num_subscription_pauses", sort_desc=False, title="Tasa de churn por número de pausas")
pauses_table

,num_subscription_pauses,churn_pct
0,0,43.027244
1,1,42.662566
2,2,42.092584
3,3,64.884971
4,4,64.292296


### 5.5 Song skip rate

Este gráfico refuerza la idea de menor ajuste o menor satisfacción con el contenido.

In [12]:
skip_table = plot_churn_by_numeric_bins(df, "song_skip_rate", "skip_rate_bin", title="Tasa de churn por intervalos de song_skip_rate")
skip_table

,skip_rate_bin,churn_pct
0,0-0.2,44.792542
1,0.2-0.4,44.430171
2,0.4-0.6,44.952618
3,0.6-0.8,55.783560
4,0.8-1.0,66.610879


### 5.6 Edad por tramos

La edad no suele comportarse de forma lineal, así que por tramos se interpreta mejor.

In [13]:
age_table = plot_churn_by_numeric_bins(df, "age", "age_group", title="Tasa de churn por grupos de edad")
age_table

,age_group,churn_pct
0,18-24,60.291764
1,25-34,50.159729
2,35-49,40.783793
3,50-64,47.698151
4,65-79,62.283614


### 5.7 Interacción clave: plan × incidencias

Este heatmap es especialmente útil porque combina dos de las variables más importantes del dataset.

In [14]:
segment_heatmap = plot_churn_heatmap(
    df,
    "subscription_type",
    "customer_service_inquiries",
    title="Heatmap de churn (%) entre subscription_type y customer_service_inquiries"
)
segment_heatmap

customer_service_inquiries,High,Low,Medium
subscription_type,,,
Family,59.335779,12.338733,31.975012
Free,96.430276,58.527280,83.343035
Premium,58.358750,13.177561,30.278884
Student,83.248830,31.496812,58.055028


### 5.8 Distribución de `weekly_hours` según churn

Dejo un gráfico de distribución para ilustrar visualmente la separación entre churn / no churn, pero sin sobrecargar con muchos histogramas o boxplots redundantes.

In [15]:
plot_distribution_by_churn(df, "weekly_hours", nbins=30, title="Distribución de weekly_hours según churn")

## 6. Geografía

Aquí dejo una función automática: si `location` contiene estados de EE. UU., dibuja un **choropleth**; si contiene regiones, dibuja un **mapa de burbujas** por región.

In [54]:
def plot_us_subscribers_map(df, state_col="state_code"):
    plot_df = (
        df.groupby(["location", state_col])
        .size()
        .reset_index(name="n_subscribers")
    )

    fig = px.choropleth(
        plot_df,
        locations=state_col,
        locationmode="USA-states",
        color="n_subscribers",
        scope="usa",
        color_continuous_scale="Burg",
        hover_name="location",
        hover_data={"state_code": False, "n_subscribers": True},
        title="Número de suscriptores por estado"
    )

    fig.update_layout(
        title_x=0.5,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=20, r=20, t=60, b=20),
        coloraxis_colorbar_title="Suscriptores"
    )

    fig.update_geos(
        bgcolor="rgba(0,0,0,0)",
        showland=True,
        landcolor="rgb(245,245,245)"
    )

    return fig

In [55]:
df_model = add_features(df)
fig_map = plot_us_subscribers_map(df_model)
fig_map.show()

In [56]:
def plot_us_churn_map(df, state_col="state_code", target="churned"):
    plot_df = (
        df.groupby(["location", state_col])[target]
        .mean()
        .reset_index(name="churn_rate")
    )

    fig = px.choropleth(
        plot_df,
        locations=state_col,
        locationmode="USA-states",
        color="churn_rate",
        scope="usa",
        color_continuous_scale="Reds",
        hover_name="location",
        hover_data={"state_code": False, "churn_rate": ":.2%"},
        title="Tasa de churn por estado"
    )

    fig.update_layout(
        title_x=0.5,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=20, r=20, t=60, b=20),
        coloraxis_colorbar_title="Churn rate"
    )

    fig.update_geos(
        bgcolor="rgba(0,0,0,0)",
        showland=True,
        landcolor="rgb(245,245,245)"
    )

    return fig

In [41]:
fig_churn = plot_us_churn_map(df_model)
fig_churn.show()

## 7. Tablas resumen útiles para la defensa y para Streamlit

In [18]:

segment_summary = (
    df.groupby(["subscription_type", "customer_service_inquiries"], as_index=False)
    .agg(
        churn_rate=("churned", "mean"),
        avg_weekly_hours=("weekly_hours", "mean"),
        avg_skip_rate=("song_skip_rate", "mean"),
        avg_pauses=("num_subscription_pauses", "mean"),
        n_customers=("churned", "size")
    )
    .sort_values("churn_rate", ascending=False)
)

segment_summary


,subscription_type,customer_service_inquiries,churn_rate,avg_weekly_hours,avg_skip_rate,avg_pauses,n_customers
3,Free,High,0.964303,24.908013,0.498994,1.999236,10477
5,Free,Medium,0.833430,24.893044,0.500347,1.978754,10308
9,Student,High,0.832488,24.762633,0.504746,1.987617,10256
0,Family,High,0.593358,24.889930,0.496553,2.002800,10358
4,Free,Low,0.585273,24.967511,0.501083,1.986265,10484
6,Premium,High,0.583587,25.205821,0.498800,1.987038,10492
11,Student,Medium,0.580550,24.991267,0.502239,1.992884,10540
2,Family,Medium,0.319750,25.212690,0.503026,2.004805,10405
10,Student,Low,0.314968,25.317156,0.504631,1.967456,10509
8,Premium,Medium,0.302789,25.309762,0.495693,1.996016,10291


## 8. Conclusiones recomendadas para contar en la defensa


### Conclusiones preliminares del EDA

A partir del análisis exploratorio, las señales más fuertes de churn parecen concentrarse en tres grandes bloques:

1. **Tipo de usuario**  
   El tipo de suscripción marca diferencias muy claras de churn, especialmente entre usuarios **Free** y planes de pago.

2. **Engagement**  
   Variables como `weekly_hours` y `song_skip_rate` sugieren que los usuarios menos activos o con peor interacción con el contenido presentan mayor riesgo de abandono.

3. **Fricción / inestabilidad**  
   Variables como `customer_service_inquiries` y `num_subscription_pauses` parecen actuar como señales tempranas de churn.

Además, el heatmap por segmentos muestra que algunas combinaciones de variables, como determinados planes junto con alta fricción, generan perfiles especialmente vulnerables.


## 9. Qué he quitado respecto al notebook original

- parte del bloque de outliers demasiado largo para la historia principal
- gráficos redundantes de la misma variable en varios formatos
- exploraciones con poco valor narrativo para la defensa
- análisis geográfico muy disperso en varias subsecciones

Si quieres, este notebook ya queda mucho mejor como base para **Streamlit** y para la **presentación oral**.